Import the libraries needed for data handling, visualization, and neural network modeling.

In [122]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler , MinMaxScaler
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
from sklearn.metrics import accuracy_score

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


Load the customer churn dataset from the project data folder so the model can learn from real records.

In [123]:
# if it is an old .xls file
df = pd.read_csv("../../datasets/Churn_Modelling.csv")

Display the first few rows of the dataset to confirm the data loaded correctly.

In [124]:
df.head(5)

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


Check the dataset shape to confirm the number of rows and columns before modeling.

In [125]:
df.shape

(10000, 14)

List the column names so the target variable and feature columns are easy to identify.

In [126]:
df.columns

Index(['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography',
       'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary', 'Exited'],
      dtype='str')

Inspect the dataset types and missing values to ensure the data is ready for training.

In [127]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  str    
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  str    
 5   Gender           10000 non-null  str    
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), str(3)
memory usage: 1.2 MB


Check whether any rows are duplicated so the model is not biased by repeated records.

In [128]:
df.isnull().sum()

RowNumber          0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

View the class distribution of the target variable to understand churn vs non-churn balance.

In [129]:
df.duplicated().sum()

np.int64(0)

Remove columns that are not useful for prediction and keep only the relevant features.

In [130]:
df["Exited"].value_counts()

Exited
0    7963
1    2037
Name: count, dtype: int64

Convert categorical variables into numeric form using one-hot encoding for machine learning.

In [131]:
df =df.drop(columns = ["RowNumber" , "CustomerId" , "Surname"])
#dropping the unnecessary columns;

Split the data into training and testing sets so the model can be evaluated honestly.

In [132]:
df = pd.get_dummies(df ,columns  = ["Geography" , "Gender"] , drop_first= True)
#applying one hot encoding to categorical columns

Confirm the training and testing sets have the expected dimensions after the split.

In [133]:
X = df.drop(columns = ["Exited"])
Y = df["Exited"]
X_train , X_test , Y_train , Y_test = train_test_split(X , Y , test_size=0.2, random_state=42)

Check the output label shape to verify it matches the training data.

In [134]:
X_train.shape

(8000, 11)

Inspect the label vector to verify the target matches the number of training rows.

In [135]:
Y_train.shape

(8000,)

Standardize the input features so the neural network trains more reliably.

In [136]:
scaler = StandardScaler()#creating an scaler object
X_train_scaled = scaler.fit_transform(X_train)#scaling the input features
X_test_scaled =  scaler.transform(X_test)#scaling the output features

Create the neural network architecture with input and hidden layers for binary classification.

In [137]:
model = Sequential()#creating the model and object
model.add(Dense(4 , activation="sigmoid" ,input_dim = 11))
#this is for creating hidden layers



c:\github_repos\ml_projects\Machine_Learning_Projects\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Add the output layer that predicts whether a customer exits or stays.

In [138]:
model.add(Dense( 1 , activation="sigmoid"))

Display the model structure to verify the layer sizes and parameter counts.

In [139]:
model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_10 (Dense)                │ (None, 4)              │            48 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │             5 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 53 (212.00 B)

 Trainable params: 53 (212.00 B)

 Non-trainable params: 0 (0.00 B)

Compile the model with a binary cross-entropy loss and Adam optimizer for training.

In [140]:
model.compile(loss = "binary_crossentropy" , optimizer="Adam")
#i dont know this line

Train the neural network on the scaled training data and learn churn patterns over several epochs.

In [141]:
model.fit(X_train_scaled , Y_train , epochs=10)

Epoch 1/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.5002
Epoch 2/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.4680
Epoch 3/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4526
Epoch 4/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4428
Epoch 5/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4367
Epoch 6/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.4328
Epoch 7/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4304
Epoch 8/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4285
Epoch 9/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.4268
Epoch 10/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4252


Inspect the learned model parameters to verify the network was initialized and trained.

In [142]:
model.layers[0].get_weights()

[array([[ 0.0749891 , -0.02342511,  0.27088258, -0.13468036],
        [-1.5676731 ,  0.681282  , -0.89183426, -0.40589058],
        [ 0.15138371, -0.16451336, -0.2042398 ,  0.07617652],
        [-0.34567922, -0.26287505,  0.17387313, -0.7778647 ],
        [-0.03267824, -0.4972583 ,  0.03486719, -0.13350588],
        [-0.01534619, -0.2614121 , -0.08945299, -0.14734662],
        [ 0.1869507 , -0.9062366 ,  0.20087437,  0.8700704 ],
        [-0.09700414,  0.06242559,  0.43805942, -0.4177434 ],
        [-0.43600866,  0.7215813 ,  0.10574339, -0.52912545],
        [-0.08711391, -0.22105725, -0.33061343,  0.09517767],
        [-0.1128718 , -0.34520394,  0.5433397 ,  0.55410963]],
       dtype=float32),
 array([ 0.17390363, -0.5163176 ,  0.03423317,  0.303657  ], dtype=float32)]

Use the trained model to predict churn probabilities for the test set.

In [143]:
Y_pred = model.predict(X_test_scaled)#predicting the output

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step


Convert the probability predictions into binary labels using a 0.5 decision threshold.

In [146]:
#now i have to make these outputs from 0 to 1 so;
#lets keep out threshold to be 0.5
Y_pred = np.where(Y_pred > 0.5 , 1 , 0)


Measure accuracy to evaluate how well the model predicts customer churn.

In [147]:
accuracy_score(Y_test , Y_pred)
# print(accuracy)

0.8165

This empty cell is a placeholder for optional final notes or additional analysis.